# Appendix A: initial coefficients by Nelson's method
For **Recurrence formulas for Taylor expansions at CM points on genus-one and genus-two Shimura curves**, by Jingxuan Geng.

Run all cells with a **SageMath** kernel. No other package file is needed. This is a shorter Sage implementation of the supplied Nelson evaluator: Sage handles quaternion arithmetic, Hermite normal forms and elliptic-curve coefficients. The lattice sums are grouped exactly in quadratic integer pairs before numerical evaluation.

The two squared lattice cutoffs are 144 and 196, with approximately 100 working decimal digits and an 80-digit weight target. The comparison threshold is $10^{-40}$. Reported differences measure **observed numerical agreement**, not an interval bound for the omitted infinite sum.

For Nelson's Theorems 3.1 and 3.9, put $r=2\mu-1$, $r^2=D$, and choose $j_0$ orthogonal to the CM field. Write $\beta=\kappa_1+\kappa_2j_0$. The exact groups implement
$$S'_n=\sum_{d\mid N}\frac1{d\,a_f(d)}
\sum_{\beta\in R+(N/d)R^*,\,\operatorname{nrd}\beta>0}
a_f(d\operatorname{nrd}\beta)
\left(\frac{X}{|X|}\right)^{2+n}\overline{\iota(\kappa_2)}^{\,n}
W_{2+n}(4\pi|X|),$$
where $X=\iota(\kappa_1)$. Consequently $c_n=S'_n(S'_0)^{n-1}/(S'_1)^n$. These values are computed independently of the recurrence.

Reference: P. D. Nelson, *Evaluating modular forms on Shimura curves*, Math. Comp. 84 (2015), [arXiv:1210.1243](https://arxiv.org/abs/1210.1243).

In [1]:
# N: (a,b,D, Z-basis of the maximal order, CM element mu).
CM_DATA = {
 14: (7, -2, -11, [[1/2, 1/2, 0, 1/2], [0, 1, 0, 0], [0, 0, 1/2, 1/2], [0, 0, 0, 1]],
      [1/2, -1/2, -3/2, 0]),
 15: (15, -30, -7, [[1/2, 0, 1/4, 1/20], [0, 1/2, 1/4, 1/60], [0, 0, 1/2, 1/30], [0, 0, 0, 1/15]],
      [1/2, 0, -1/4, 1/60]),
 21: (21, -1, -43, [[1/2, 1/2, 0, 0], [0, 1, 0, 0], [0, 0, 1/2, 1/2], [0, 0, 0, 1]],
      [1/2, -1/2, -10, -2]),
 33: (33, -1, -67, [[1/2, 1/2, 0, 0], [0, 1, 0, 0], [0, 0, 1/2, 1/2], [0, 0, 0, 1]],
      [1/2, -1/2, -5, 0]),
 34: (34, -3, -11, [[1/2, 0, 1/6, 1/3], [0, 1/2, 0, 1/2], [0, 0, 1/3, 2/3], [0, 0, 0, 1]],
      [1/2, 1/2, -7/2, 1/2]),
 46: (46, -3, -163, [[1/2, 0, 1/6, 1/3], [0, 1/2, 0, 1/2], [0, 0, 1/3, 2/3], [0, 0, 0, 1]],
      [1/2, 0, -35/6, -2/3]),
 26: (26, -5, -11, [[1/2, 0, 1/10, 1/10], [0, 1/2, 0, 1/2], [0, 0, 1/5, 1/5], [0, 0, 0, 1]],
      [1/2, 1, -7/2, -1/2]),
 38: (-19, 2, -11, [[1/2, 1/2, 0, 0], [0, 1, 0, 0], [0, 0, 1/2, 1/2], [0, 0, 0, 1]],
      [1/2, -1/2, -1, 0]),
 58: (-29, 2, -11, [[1/2, 1/2, 0, 1/2], [0, 1, 0, 0], [0, 0, 1/2, 1/2], [0, 0, 0, 1]],
      [1/2, 1/2, 3/2, 0]),
}

## Exact quaternion and lattice arithmetic
The order check verifies containment of 1, multiplication closure and reduced discriminant $N$. The CM embedding and upper-half-plane fixed point are checked exactly.

In [2]:
from sage.all import *
from collections import defaultdict
from itertools import product
from math import isqrt, cos, exp, log, gamma, pi

RF = RealField(336)                 # about 100 decimal digits
CF = ComplexField(336)

def lattice(rows):
    """A row basis of the exact Z-span, in upper triangular Hermite form."""
    M = matrix(QQ, rows)
    d = lcm([x.denominator() for x in M.list()])
    return (d*M).change_ring(ZZ).hermite_form(include_zero_rows=False)/d

def embedding(a, b, q):
    x, y, z, w = q
    if a < 0:
        a, b, y, z, w = b, a, z, y, -w
    p, s = QQbar(a).sqrt(), QQbar(-b).sqrt()
    return matrix(QQbar, [[x+y*p, (z+w*p)*s], [(-z+w*p)*s, x-y*p]])

class CMPoint:
    """Exact order/CM inputs and the split lattices in Nelson's formula."""
    def __init__(self, N):
        a, b, D, basis, mu = CM_DATA[N]
        self.N, self.D = N, D
        B = QuaternionAlgebra(QQ, a, b)
        R, mu = matrix(QQ, basis), B(list(map(QQ, mu)))
        inside = lambda q: all(x in ZZ for x in vector(QQ, list(q))*R.inverse())
        rb = [B(list(row)) for row in R]
        assert B.discriminant() == N and inside(B(1)) and inside(mu)
        assert all(inside(x*y) for x in rb for y in rb)
        assert all(q.reduced_trace() in ZZ and q.reduced_norm() in ZZ for q in rb)
        T = matrix(QQ, [[(x*y.conjugate()).reduced_trace() for y in rb] for x in rb])
        assert abs(T.det()) == N*N  # an integral order of this discriminant is maximal
        assert mu.reduced_trace() == 1 and 1-4*mu.reduced_norm() == D
        assert QuadraticField(D).discriminant() == D and QuadraticField(D).class_number() == 1
        M = embedding(a, b, mu)
        sign = 1 if M[1,0].real() > 0 else -1
        self.tau = (M[0,0]-M[1,1]+sign*QQbar(D).sqrt())/(2*M[1,0])
        assert self.tau.imag() > 0
        assert M[1,0]*self.tau**2+(M[1,1]-M[0,0])*self.tau-M[0,1] == 0
        r = 2*mu-1
        candidates = [B([0]+list(v)) for v in product(range(-3,4), repeat=3) if any(v)]
        candidates = [v for v in candidates if (v*mu.conjugate()).reduced_trace()==0]
        if not candidates:
            v = (-a*mu[1],-b*mu[2],a*b*mu[3])
            candidates = [q for q in [B([0,-v[1],v[0],0]),B([0,-v[2],0,v[0]]),
                                      B([0,0,-v[2],v[1]])] if q]
            candidates = candidates[:1]  # exact fallback used in the supplied evaluator
        j = min(candidates,key=lambda v: abs(v.reduced_norm()))
        self.c = -j.reduced_norm()
        assert self.c > 0 and j*r == -r*j
        C = matrix(QQ, [list(q) for q in [B(1),r,j,r*j]]).inverse()
        x, y = CF(self.tau).real(), CF(self.tau).imag()
        sigma = matrix(CF, [[y.sqrt(), x/y.sqrt()], [0,1/y.sqrt()]])
        def XY(q):
            U = sigma.inverse()*embedding(a,b,q).change_ring(CF)*sigma
            return ((U[0,0]+U[1,1]+CF.gen()*(U[0,1]-U[1,0]))/2,
                    (U[0,0]-U[1,1]+CF.gen()*(U[0,1]+U[1,0]))/2)
        Xr, _ = XY(r)
        self.eps = 1 if Xr.imag()>0 else -1
        Xj, self.Y = XY(j)
        assert abs(Xj)<RF('1e-90') and abs(abs(self.Y)**2-self.c)<RF('1e-90')
        dual = T.inverse()*R
        self.split = {}
        for d in divisors(N):
            V = lattice(list(R.rows())+list(((N//d)*dual).rows()))*C
            den = lcm([x.denominator() for x in V.list()])
            H = (den*V).change_ring(ZZ).hermite_form()
            self.split[int(d)] = (int(den), [list(map(int,row)) for row in H])

    def grouped_sums(self, d, an, cutoff, nmax):
        """Exact integer-pair sums in Z[r], r^2=D; cutoff bounds |kappa_1|^2."""
        L, H = self.split[d]
        D, p = int(self.D), -int(self.D)
        cn, cd = int(self.c.numerator()), int(self.c.denominator())
        (h11,h12,h13,h14),(_,h22,h23,h24),(_,_,h33,h34),(_,_,_,h44) = H
        bound = int(cutoff)*L*L
        def powers(x,y,start,count):
            u,v = 1,0
            for n in range(start+count):
                if n>=start: yield (u,v)
                u,v = u*x+D*v*y,u*y+v*x
        pairs = lambda: [[0,0] for _ in range(nmax+1)]
        classes = defaultdict(lambda: defaultdict(pairs))
        for a in range(-isqrt(bound)//h11-1, isqrt(bound)//h11+2):
            x = a*h11
            if x*x>bound: continue
            t = isqrt((bound-x*x)//p)+1
            for b in range((-t-a*h12)//h22-1, (t-a*h12)//h22+2):
                y = a*h12+b*h22
                k = x*x+p*y*y
                if not 0<k<=bound: continue
                s,t = a*h13+b*h23,a*h14+b*h24
                q = s//h33
                acc = classes[(s-q*h33,(t-q*h34)%h44)][k]
                for n,(u,v) in enumerate(powers(x,y,2,nmax+1)):
                    acc[n][0]+=u; acc[n][1]+=v
        G = defaultdict(pairs)
        for (s,t),rows in classes.items():
            bound2 = (cd*max(rows)-1)//cn
            second = defaultdict(pairs)
            for a in range(-isqrt(bound2)//h33-2, isqrt(bound2)//h33+3):
                x = s+a*h33
                if x*x>bound2: continue
                bnd = isqrt((bound2-x*x)//p)+1
                base = t+a*h34
                for b in range((-bnd-base)//h44-1,(bnd-base)//h44+2):
                    y = base+b*h44
                    k = x*x+p*y*y
                    if k>bound2: continue
                    acc = second[k]
                    for n,(u,v) in enumerate(powers(x,-y,0,nmax+1)):
                        acc[n][0]+=u; acc[n][1]+=v
            for k,J in rows.items():
                I = pairs()
                for l in sorted(second):
                    norm = cd*k-cn*l
                    if norm<=0: break
                    index,rem = divmod(d*norm,cd*L*L)
                    assert rem==0
                    for n,(u,v) in enumerate(second[l]):
                        I[n][0]+=an[index]*u; I[n][1]+=an[index]*v
                for n,((a,b),(c,e)) in enumerate(zip(J,I)):
                    G[k][n][0]+=a*c+D*b*e; G[k][n][1]+=a*e+b*c
        return L,G

## Weight functions and finite sums
The weight integral is
$$W_k(x)=2^{1-k}\int_0^\infty\bigl[x\cosh((k-1)t)\operatorname{Li}_{-k-1}(e^{-x\cosh t})-\cosh(kt)\operatorname{Li}_{-k}(e^{-x\cosh t})\bigr]dt.$$
The supplied trapezoidal step and tail heuristic is retained. The exact $a_f(n)$ come from Sage's elliptic-curve database.

In [3]:
def weight(x, ks, digits=80):
    """Trapezoidal evaluation of Nelson's Bessel-weight integral.

    The same step/tail heuristic as the supplied evaluator; no interval bound.
    Li_{-s}(q) = sum_k k! S(s+1,k+1) (q/(1-q))^(k+1).
    """
    n,x0 = max(ks),float(x)
    xa = x0*cos(1.5); rho = exp(-xa/2)
    majorant = log(2*gamma(n)*2**(2*n-1))-n*log(xa)+log(x0*rho/(1-rho)**2+rho/(1-rho))
    majorant = min(majorant,log(2*(x0+1))+0.5*log(2*pi/xa)+n*n/(2*xa)-xa+30)
    log_eps = -digits*log(10)-x0
    h = RF(str(2*pi*1.5/(majorant-log_eps+2)))
    end = 0.0
    while log(4/3*(x0+1))+n*end-x0*float(cosh(end))>log_eps-10 or x0*float(cosh(end))<30:
        end+=0.02
    coeff = {s:[factorial(k)*stirling_number2(s+1,k+1) for k in range(s+1)]
             for s in range(min(ks),n+2)}
    total = {k:RF(0) for k in ks}
    for j in range(int(end/float(h))+3):
        e = (j*h).exp(); ei = 1/e
        q = (-x*(e+ei)/2).exp(); r = q/(1-q)
        powers = [r**k for k in range(n+3)]
        Li = {s:sum(c*powers[k+1] for k,c in enumerate(coeff[s])) for s in coeff}
        ch = [(e**k+ei**k)/2 for k in range(n+1)]
        for k in ks:
            total[k]+=(RF(1)/2 if j==0 else 1)*(x*ch[k-1]*Li[k+1]-ch[k]*Li[k])
    return {k:total[k]*h/2**(k-1) for k in ks}

def nelson(form, point, nmax=1, cutoff=196):
    """Return normalized c_2,...,c_nmax and Theta from finite Nelson sums."""
    N = point.N
    an = list(map(int,EllipticCurve(form+'1').anlist(int(N*cutoff+10))))
    sums = [CF(0) for _ in range(nmax+1)]
    W = {}
    for d in divisors(N):
        L,G = point.grouped_sums(int(d),an,int(cutoff),int(nmax))
        for k,terms in G.items():
            m = QQ(k)/(L*L)
            if m not in W: W[m]=weight(4*RF.pi()*RF(m).sqrt(),range(2,nmax+3))
            for n,(a,b) in enumerate(terms):
                term = CF(a,point.eps*b*RF(-point.D).sqrt())
                sums[n]+=term*W[m][n+2]/(d*an[d]*k*RF(L)**n*RF(k).sqrt()**n)
    assert sums[0] and sums[1]
    c = {n:sums[n]*sums[0]**(n-1)/sums[1]**n for n in range(2,nmax+1)}
    theta = -4*RF.pi()*point.Y*sums[1]/sums[0]
    return c,theta

In [4]:
# Appendix A: indices in Table 6; values in Table 7 (c_0=c_1=1).
initial = {
 '14a': [-4, 214/9],
 '15a': [-1/4, 27/16],
 '21a': [-905/3969, -1318/27783],
 '33a': [-72337/529, 9202302/12167, 3009372468/279841, -1149714703920/6436343],
 '34a': [-12/25, 2762/125, -17736/625, -67648/125],
 '46a': [27824049/123402050, -114816966372/484661551375, -784666539922896/3807016486050625, -3694325348311426944/5980822899585531875],
 '26a': [-16/147, 274/3087, -2488/9261, 352768/1361367],
 '26b': [40, -734, -11240, 1308544],
 '38a': [-64/27, -62/27, -19672/2187, 41984/6561],
 '38b': [-16/169, -878/2197, 7688/28561, -2699648/371293],
 '58a': [-4/5, 314/125, 31192/625, -1977728/625, 4428752/3125],
 '58b': [140/9, 262/3, -39592/81, 420224/81],
}

In [5]:
cutoffs = (144,196)
tolerance = RF('1e-40')
points = {}
rows,summary = [],[]
for form,targets in initial.items():
    N = ZZ(form[:-1])
    if N not in points: points[N] = CMPoint(N)
    results = [nelson(form,points[N],len(targets)+1,M)[0] for M in cutoffs]
    differences = [[abs(values[n]-target) for n,target in enumerate(targets,2)] for values in results]
    assert all(d<tolerance for ds in differences for d in ds), form
    change = max(abs(results[1][n]-results[0][n]) for n in results[1])
    summary.append([form,max(differences[0]),max(differences[1]),change])
    for n,target in enumerate(targets,2):
        rows.append([form,n,target,results[1][n],abs(results[1][n]-target)])
    print(form, 'PASS; largest observed difference at 196:',max(differences[1]),flush=True)
assert len(initial)==12 and len(rows)==43
show(table(rows,header_row=['form','n','stated c_n','computed c_n (cutoff 196)','observed difference']))
show(table(summary,header_row=['form','max difference at 144','max difference at 196','cutoff change']))
print('All 43 initial coefficients satisfy the requested numerical comparison.')
print('Largest observed difference at 196:',max(row[-1] for row in rows))


14a PASS; largest observed difference at 196: 8.985769612091561470471119863118849987571693055336387852836931426109019595701483169518298111356202691e-65


15a PASS; largest observed difference at 196: 5.004839026771731996788755620366982863313455249450795871824952896078390135185026657726632112154936414e-66


21a PASS; largest observed difference at 196: 4.125624193008147217249699701276926342851162139349288738063655157746899744867382198646867809708710747e-68


33a PASS; largest observed difference at 196: 9.095397267330845985543428699116416597105106390447498097796632056907528069240577516868992019310172224e-59


34a PASS; largest observed difference at 196: 4.299388547716105199693770966638529928267458620156638804623456616115743492307944994119308397369261122e-61


46a PASS; largest observed difference at 196: 4.289696732503699384824383850738861122337069483494422523866144695809583771292222113608445463880760891e-64


... (5 outputs left out)

58b PASS; largest observed difference at 196: 7.781354899245516501520047461322335326734304587556225326289859223643869479018155450897312442387421829e-61


form,n,stated c_n,computed c_n (cutoff 196),observed difference
14a,,,,
14a,,,,
15a,,,,
15a,,,,
21a,,,,
21a,,,,
33a,,,,
33a,,,,
33a,,,,
33a,,,,


form,max difference at 144,max difference at 196,cutoff change
14a,,,
15a,,,
21a,,,
33a,,,
34a,,,
46a,,,
26a,,,
26b,,,
38a,,,
38b,,,


All 43 initial coefficients satisfy the requested numerical comparison.
Largest observed difference at 196: 1.207268332608483331225310369126332248135852735507859427829028350589421759819406278860115864851009114e-57
